In [24]:
import pandas as pd
from urllib.parse import urlparse


В этом датасете, мы возьмём данные о категории сайта.

Категория домена будет опциональным признаком в итоговом датасете(не все тон домены имеют схожие сайты). Соответсвенно одно наличие этого навыка будет поднимать цену домену.
В связи с этим возьмём только популярные категории

In [ ]:
curlie=pd.read_csv('../raw_datasets/curlie_classification.csv',
                   usecols=[0,1], 
                   names=['domain', 'macro_topic'], 
                   header=0, 
                   engine='python')

In [26]:
curlie.shape

(229648, 2)

Сразу видим что в датасете есть домены с неверным форматом.

In [27]:
curlie=curlie[curlie['domain'].str.match(r'^[a-zA-Z0-9\.-]+$')]
top_categories=curlie['macro_topic'].value_counts().reset_index().head(20)['macro_topic']

In [28]:
top_curlie= curlie[curlie['macro_topic'].isin(top_categories)]
top_curlie['domain']=top_curlie['domain'].map(lambda x: x.split('.')[0])


In [29]:
top_curlie.shape

(225532, 2)

Kaggle класификация сайтов

In [ ]:
def extract_domain(url):
    try:
        netloc = urlparse(url).netloc.lower()
        if netloc.startswith('www.'):
            netloc = netloc[4:]
        parts = netloc.split('.')
        if len(parts) >= 3:
            return None
        else:
            return parts[0]
    except:
        return None
kaggle=pd.read_csv('../raw_datasets/kaggle_url_classification.csv', 
                   index_col=0, 
                   header=None, 
                   names=['domain','category']
)
kaggle['domain']=kaggle['domain'].apply(extract_domain)

In [31]:
kaggle=kaggle[kaggle['domain'].notna()]

Функция не обрабатывает домены с несколькими точками в имени такие как:

rss.cnn.com
или 
scotwrestle.co.uk


мы их не можем грамотно обработать, и их значений не много в сравнении всего датасета. Поэтому принято решение их отбросить 


In [32]:
kaggle=kaggle[kaggle['domain'].isna()]

Посмотрим какие категории есть в датасете

In [33]:
kaggle['category'].value_counts().reset_index()['category']

Series([], Name: category, dtype: str)

Всего 15 категорий
Сравним их с топовыми категориями из предыдущего датасета

In [34]:
# (set(kaggle['category'].value_counts().reset_index()['category']) -set(top_categories))

Противоречий не найдено, можно их объединять в итоговый датафрейм 

In [35]:
res_categories=pd.concat([kaggle, top_curlie.rename(columns={'macro_topic':'category'})])

In [36]:
res_categories.isna().sum()

domain      0
category    0
dtype: int64

In [37]:
res_categories=res_categories.drop_duplicates(subset='domain')

In [ ]:
res_categories.to_csv('../datasets/domens_categories.csv', index=False)